## MODELO 3 CAPAS

In [1]:
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "./data"))
MODELS_DIR = Path(os.environ.get("MODELS_DIR", "./models"))
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, Conv2D, MaxPool2D, Flatten, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# ============================================================
# PASO 0. RUTAS DE LAS CLASES
# ============================================================
ruta_perros = DATA_DIR / "OXFORD" / "PERROS"
ruta_gatos = DATA_DIR / "OXFORD" / "GATOS"

# UNA SOLA RUTA PARA EL MEJOR MODELO
ruta_modelo = MODELS_DIR / "3capas_modelo_fase1_perro_gato.keras"

IMG_HEIGHT = 224
IMG_WIDTH = 224

classes = [
    ("PERRO", ruta_perros),
    ("GATO", ruta_gatos)
]

# ============================================================
# PASO 1. CARGA DE IMÁGENES
# ============================================================
X = []
y = []

for class_index, (class_name, class_folder) in enumerate(classes):
    for root, _, files in os.walk(class_folder):
        for file_name in files:
            if not file_name.lower().endswith((".png", ".jpg", ".jpeg")):
                continue

            file_path = os.path.join(root, file_name)

            try:
                img = tf.keras.utils.load_img(
                    file_path,
                    color_mode="rgb",
                    target_size=(IMG_HEIGHT, IMG_WIDTH)
                )

                img_array = tf.keras.utils.img_to_array(img)

                X.append(img_array)
                y.append(class_index)

            except Exception as e:
                print(f"Error al leer {file_path}: {e}")

X = np.array(X, dtype="float32")
y = np.array(y)

print("Shape total de X:", X.shape)
print("Shape total de y:", y.shape)
print("Total PERROS:", np.sum(y == 0))
print("Total GATOS:", np.sum(y == 1))

# ============================================================
# PASO 2. PREPROCESAMIENTO
# ============================================================
X /= 255.0

# 70 / 15 / 15
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("\nDistribución final:")
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nClases por conjunto:")
print("Train -> PERROS:", np.sum(y_train == 0), " | GATOS:", np.sum(y_train == 1))
print("Validation -> PERROS:", np.sum(y_val == 0), " | GATOS:", np.sum(y_val == 1))
print("Test -> PERROS:", np.sum(y_test == 0), " | GATOS:", np.sum(y_test == 1))

# ============================================================
# PASO 3. CLASS WEIGHT
# ============================================================
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = {
    0: class_weights[0],
    1: class_weights[1]
}

print("\nClass weights:", class_weight_dict)

# ============================================================
# PASO 4. DATA AUGMENTATION
# ============================================================
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10)
])

# ============================================================
# PASO 5. MODELO
# ============================================================
model = Sequential()

model.add(tf.keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)))
model.add(data_augmentation)

# Bloque 1
model.add(Conv2D(32, (3, 3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(MaxPool2D((2, 2)))

# Bloque 2
model.add(Conv2D(64, (3, 3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(MaxPool2D((2, 2)))

# Bloque 3
model.add(Conv2D(128, (3, 3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(MaxPool2D((2, 2)))

# Regularización
model.add(Dropout(0.30))

# FLATTEN
model.add(Flatten())

# Capa densa
model.add(Dense(128, activation="relu"))
model.add(Dropout(0.50))

# Salida
model.add(Dense(1, activation="sigmoid"))

# ============================================================
# PASO 6. COMPILACIÓN
# ============================================================
model.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    metrics=["accuracy"]
)

model.summary()

# ============================================================
# PASO 7. CALLBACKS
# ============================================================
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath=ruta_modelo,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    mode="min",
    verbose=1
)

# ============================================================
# PASO 8. ENTRENAMIENTO
# ============================================================
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=30,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr, checkpoint],
    class_weight=class_weight_dict,
    verbose=1
)

# ============================================================
# PASO 8.1 CARGAR EL MEJOR MODELO
# ============================================================
print("\nCargando el mejor modelo guardado...")
model = load_model(ruta_modelo)

# ============================================================
# PASO 9. EVALUACIÓN FINAL EN TEST
# ============================================================
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("\nTest loss:", test_loss)
print("Test accuracy:", test_acc)

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob >= 0.50).astype(int).ravel()

acc = accuracy_score(y_test, y_pred)
print("Accuracy score (sklearn):", acc)

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred, target_names=["PERRO", "GATO"]))

# ============================================================
# PASO 10. MODELO FINAL
# ============================================================
print("\nEl mejor modelo quedó guardado en:", ruta_modelo)

Shape total de X: (7390, 224, 224, 3)
Shape total de y: (7390,)
Total PERROS: 4990
Total GATOS: 2400

Distribución final:
Train: (5173, 224, 224, 3) (5173,)
Validation: (1108, 224, 224, 3) (1108,)
Test: (1109, 224, 224, 3) (1109,)

Clases por conjunto:
Train -> PERROS: 3493  | GATOS: 1680
Validation -> PERROS: 748  | GATOS: 360
Test -> PERROS: 749  | GATOS: 360

Class weights: {0: np.float64(0.7404809619238477), 1: np.float64(1.5395833333333333)}


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential (Sequential)              │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 224, 224, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 224, 224, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 112, 112, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 112, 112, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 112, 112, 64)        │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 56, 56, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 56, 56, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 56, 56, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 28, 28, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 28, 28, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 100352)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │      12,845,184 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 12,939,457 (49.36 MB)

 Trainable params: 12,939,009 (49.36 MB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/30
162/162 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5817 - loss: 1.2135
Epoch 1: val_loss improved from None to 0.71475, saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\3capas_modelo_fase1_perro_gato.keras

Epoch 1: finished saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\3capas_modelo_fase1_perro_gato.keras
162/162 ━━━━━━━━━━━━━━━━━━━━ 332s 2s/step - accuracy: 0.5641 - loss: 0.8986 - val_accuracy: 0.6309 - val_loss: 0.7148 - learning_rate: 1.0000e-04
Epoch 2/30
162/162 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5682 - loss: 0.6567
Epoch 2: val_loss did not improve from 0.71475
162/162 ━━━━━━━━━━━━━━━━━━━━ 319s 2s/step - accuracy: 0.6014 - loss: 0.6522 - val_accuracy: 0.6625 - val_loss: 1.1296 - learning_rate: 1.0000e-04
Epoch 3/30
162/162 ━━━━━━━━━━━━━━━━━━━━ 0s 2s

## MODELO 4 CAPAS

In [1]:
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "./data"))
MODELS_DIR = Path(os.environ.get("MODELS_DIR", "./models"))
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, Conv2D, MaxPool2D, Flatten, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# ============================================================
# PASO 0. RUTAS DE LAS CLASES
# ============================================================
ruta_perros = DATA_DIR / "OXFORD" / "PERROS"
ruta_gatos = DATA_DIR / "OXFORD" / "GATOS"

# UNA SOLA RUTA PARA EL MEJOR MODELO
ruta_modelo = MODELS_DIR / "4capas_modelo_fase1_perro_gato.keras"

IMG_HEIGHT = 224
IMG_WIDTH = 224

classes = [
    ("PERRO", ruta_perros),
    ("GATO", ruta_gatos)
]

# ============================================================
# PASO 1. CARGA DE IMÁGENES
# ============================================================
X = []
y = []

for class_index, (class_name, class_folder) in enumerate(classes):
    for root, _, files in os.walk(class_folder):
        for file_name in files:
            if not file_name.lower().endswith((".png", ".jpg", ".jpeg")):
                continue

            file_path = os.path.join(root, file_name)

            try:
                img = tf.keras.utils.load_img(
                    file_path,
                    color_mode="rgb",
                    target_size=(IMG_HEIGHT, IMG_WIDTH)
                )

                img_array = tf.keras.utils.img_to_array(img)

                X.append(img_array)
                y.append(class_index)

            except Exception as e:
                print(f"Error al leer {file_path}: {e}")

X = np.array(X, dtype="float32")
y = np.array(y)

print("Shape total de X:", X.shape)
print("Shape total de y:", y.shape)
print("Total PERROS:", np.sum(y == 0))
print("Total GATOS:", np.sum(y == 1))

# ============================================================
# PASO 2. PREPROCESAMIENTO
# ============================================================
X /= 255.0

# 70 / 15 / 15
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("\nDistribución final:")
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nClases por conjunto:")
print("Train -> PERROS:", np.sum(y_train == 0), " | GATOS:", np.sum(y_train == 1))
print("Validation -> PERROS:", np.sum(y_val == 0), " | GATOS:", np.sum(y_val == 1))
print("Test -> PERROS:", np.sum(y_test == 0), " | GATOS:", np.sum(y_test == 1))

# ============================================================
# PASO 3. CLASS WEIGHT
# ============================================================
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = {
    0: class_weights[0],
    1: class_weights[1]
}

print("\nClass weights:", class_weight_dict)

# ============================================================
# PASO 4. DATA AUGMENTATION
# ============================================================
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10)
])

# ============================================================
# PASO 5. MODELO
# ============================================================
model = Sequential()

model.add(tf.keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)))
model.add(data_augmentation)

# Bloque 1
model.add(Conv2D(32, (3, 3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(MaxPool2D((2, 2)))

# Bloque 2
model.add(Conv2D(64, (3, 3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(MaxPool2D((2, 2)))

# Bloque 3
model.add(Conv2D(128, (3, 3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(MaxPool2D((2, 2)))

# Bloque 4
model.add(Conv2D(256, (3, 3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(MaxPool2D((2, 2)))

# Regularización
model.add(Dropout(0.30))

# FLATTEN
model.add(Flatten())

# Capa densa
model.add(Dense(128, activation="relu"))
model.add(Dropout(0.50))

# Salida
model.add(Dense(1, activation="sigmoid"))

# ============================================================
# PASO 6. COMPILACIÓN
# ============================================================
model.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    metrics=["accuracy"]
)

model.summary()

# ============================================================
# PASO 7. CALLBACKS
# ============================================================
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath=ruta_modelo,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    mode="min",
    verbose=1
)

# ============================================================
# PASO 8. ENTRENAMIENTO
# ============================================================
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=30,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr, checkpoint],
    class_weight=class_weight_dict,
    verbose=1
)

# ============================================================
# PASO 8.1 CARGAR EL MEJOR MODELO
# ============================================================
print("\nCargando el mejor modelo guardado...")
model = load_model(ruta_modelo)

# ============================================================
# PASO 9. EVALUACIÓN FINAL EN TEST
# ============================================================
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("\nTest loss:", test_loss)
print("Test accuracy:", test_acc)

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob >= 0.50).astype(int).ravel()

acc = accuracy_score(y_test, y_pred)
print("Accuracy score (sklearn):", acc)

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred, target_names=["PERRO", "GATO"]))

# ============================================================
# PASO 10. MODELO FINAL
# ============================================================
print("\nEl mejor modelo quedó guardado en:", ruta_modelo)

Shape total de X: (7390, 224, 224, 3)
Shape total de y: (7390,)
Total PERROS: 4990
Total GATOS: 2400

Distribución final:
Train: (5173, 224, 224, 3) (5173,)
Validation: (1108, 224, 224, 3) (1108,)
Test: (1109, 224, 224, 3) (1109,)

Clases por conjunto:
Train -> PERROS: 3493  | GATOS: 1680
Validation -> PERROS: 748  | GATOS: 360
Test -> PERROS: 749  | GATOS: 360

Class weights: {0: np.float64(0.7404809619238477), 1: np.float64(1.5395833333333333)}


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential (Sequential)              │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 224, 224, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 224, 224, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 112, 112, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 112, 112, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 112, 112, 64)        │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 56, 56, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 56, 56, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 56, 56, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 28, 28, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 28, 28, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 28, 28, 256)         │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 14, 14, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 14, 14, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 50176)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │       6,422,656 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,813,121 (25.99 MB)

 Trainable params: 6,812,161 (25.99 MB)

 Non-trainable params: 960 (3.75 KB)

Epoch 1/30
162/162 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5779 - loss: 1.1095
Epoch 1: val_loss improved from None to 1.00040, saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\4capas_modelo_fase1_perro_gato.keras

Epoch 1: finished saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\4capas_modelo_fase1_perro_gato.keras
162/162 ━━━━━━━━━━━━━━━━━━━━ 321s 2s/step - accuracy: 0.6033 - loss: 0.8341 - val_accuracy: 0.3484 - val_loss: 1.0004 - learning_rate: 1.0000e-04
Epoch 2/30
162/162 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6624 - loss: 0.6332
Epoch 2: val_loss improved from 1.00040 to 0.74180, saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\4capas_modelo_fase1_perro_gato.ker

## MODELO 5 CAPAS

In [1]:
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "./data"))
MODELS_DIR = Path(os.environ.get("MODELS_DIR", "./models"))
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, Conv2D, MaxPool2D, Flatten, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# ============================================================
# PASO 0. RUTAS DE LAS CLASES
# ============================================================
ruta_perros = DATA_DIR / "OXFORD" / "PERROS"
ruta_gatos = DATA_DIR / "OXFORD" / "GATOS"

# UNA SOLA RUTA PARA EL MEJOR MODELO
ruta_modelo = MODELS_DIR / "5capas_modelo_fase1_perro_gato.keras"

IMG_HEIGHT = 224
IMG_WIDTH = 224

classes = [
    ("PERRO", ruta_perros),
    ("GATO", ruta_gatos)
]

# ============================================================
# PASO 1. CARGA DE IMÁGENES
# ============================================================
X = []
y = []

for class_index, (class_name, class_folder) in enumerate(classes):
    for root, _, files in os.walk(class_folder):
        for file_name in files:
            if not file_name.lower().endswith((".png", ".jpg", ".jpeg")):
                continue

            file_path = os.path.join(root, file_name)

            try:
                img = tf.keras.utils.load_img(
                    file_path,
                    color_mode="rgb",
                    target_size=(IMG_HEIGHT, IMG_WIDTH)
                )

                img_array = tf.keras.utils.img_to_array(img)

                X.append(img_array)
                y.append(class_index)

            except Exception as e:
                print(f"Error al leer {file_path}: {e}")

X = np.array(X, dtype="float32")
y = np.array(y)

print("Shape total de X:", X.shape)
print("Shape total de y:", y.shape)
print("Total PERROS:", np.sum(y == 0))
print("Total GATOS:", np.sum(y == 1))

# ============================================================
# PASO 2. PREPROCESAMIENTO
# ============================================================

# En esta parte se normalizan los valores de los pixeles dividiéndolos
# entre 255, logrando que cada valor quede en un rango de 0 a 1.
# Esto ayuda a que la red neuronal entrene de forma más estable y rápida.

X /= 255.0 

# Después, el dataset se divide en tres conjuntos:
# 70% para entrenamiento,
# 15% para validación,
# 15% para prueba final.
# random_state=42 fija una semilla aleatoria para que la división de datos
# siempre sea exactamente la misma cada vez que se ejecuta el código.

# Esto permite reproducibilidad, es decir, obtener los mismos conjuntos
# de entrenamiento, validación y prueba en futuras ejecuciones.

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,  
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("\nDistribución final:")
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nClases por conjunto:")
print("Train -> PERROS:", np.sum(y_train == 0), " | GATOS:", np.sum(y_train == 1))
print("Validation -> PERROS:", np.sum(y_val == 0), " | GATOS:", np.sum(y_val == 1))
print("Test -> PERROS:", np.sum(y_test == 0), " | GATOS:", np.sum(y_test == 1))

# ============================================================
# PASO 3. CLASS WEIGHT
# ============================================================
# Esta sección calcula pesos para cada clase con el objetivo de compensar
# posibles diferencias en la cantidad de imágenes entre perros y gatos.

# Si una clase tiene menos ejemplos, se le asigna un peso mayor para que
# el modelo le preste más atención durante el entrenamiento.

# Esto ayuda a reducir sesgos hacia la clase mayoritaria y mejora el
# aprendizaje en datasets desbalanceados.

# Finalmente, los pesos se guardan en un diccionario donde:
# 0 = perros
# 1 = gatos
# y después se muestran en pantalla.

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = {
    0: class_weights[0],
    1: class_weights[1]
}

print("\nClass weights:", class_weight_dict)

# ============================================================
# PASO 4. DATA AUGMENTATION
# ============================================================
# En esta parte se crea una etapa de Data Augmentation, la cual genera
# variaciones aleatorias de las imágenes originales durante el entrenamiento.

# Se aplican transformaciones como:
# volteo horizontal,
# ligera rotación,
# zoom,
# y cambios suaves de contraste.

# El objetivo es aumentar la diversidad visual del dataset sin recolectar
# nuevas imágenes, ayudando a que el modelo generalice mejor.

# Esto reduce el sobreajuste (overfitting) y mejora el desempeño ante
# imágenes nuevas o diferentes a las vistas en entrenamiento.

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10)
])

# ============================================================
# PASO 5. MODELO
# ============================================================
model = Sequential()   # Crea una red neuronal secuencial donde las capas se agregan una tras otra.

model.add(tf.keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)))  # Recibe imágenes de tamaño (224, 224, 3 canales RGB).
model.add(data_augmentation)  # Aplica transformaciones aleatorias a la imagen en entrenamiento.

# Bloque 1
model.add(Conv2D(32, (3, 3), padding="same", activation="relu")) # Aplica 32 filtros de tamaño 3x3. Neuronas = 224 * 224 * 32 = 1,605,632
model.add(BatchNormalization()) # Normaliza activaciones para entrenamiento más estable.
model.add(MaxPool2D((2, 2)))  # Reduce tamaño tomando el valor máximo en ventanas 2x2. # Salida: (112,112,32)

# Bloque 2
model.add(Conv2D(64, (3, 3), padding="same", activation="relu")) # Aplica 64 filtros de tamaño 3x3. Neuronas = 112 * 112 * 64 = 802,816
model.add(BatchNormalization()) # Normaliza activaciones para entrenamiento más estable.
model.add(MaxPool2D((2, 2))) # Reduce tamaño tomando el valor máximo en ventanas 2x2. # Salida: (56,56,64)

# Bloque 3
model.add(Conv2D(128, (3, 3), padding="same", activation="relu")) # Aplica 128 filtros de tamaño 3x3. Neuronas = 56 * 56 * 128 = 401,408
model.add(BatchNormalization()) # Normaliza activaciones para entrenamiento más estable.
model.add(MaxPool2D((2, 2)))  # Reduce tamaño tomando el valor máximo en ventanas 2x2. # Salida: (28,28,128)

# Bloque 4
model.add(Conv2D(256, (3, 3), padding="same", activation="relu")) # Aplica 256 filtros de tamaño 3x3. Neuronas = 28 * 28 * 256 = 200,704
model.add(BatchNormalization()) # Normaliza activaciones para entrenamiento más estable.
model.add(MaxPool2D((2, 2))) # Reduce tamaño tomando el valor máximo en ventanas 2x2. # Salida: (14,14,256)

# Bloque 5
model.add(Conv2D(512, (3, 3), padding="same", activation="relu")) # Aplica 512 filtros de tamaño 3x3. Neuronas = 14 * 14 * 512 = 100,352
model.add(BatchNormalization()) # Normaliza activaciones para entrenamiento más estable.
model.add(MaxPool2D((2, 2))) # Reduce tamaño tomando el valor máximo en ventanas 2x2. # Salida: (7,7,512)

# Regularización
model.add(Dropout(0.40))  # Apaga aleatoriamente 40% de neuronas en entrenamiento.  # Evita sobreajuste.

# FLATTEN
model.add(Flatten())  # Convierte volumen 7x7x512 en vector 1D. # Cálculo: 7*7*512 = 25088 neuronas

# Capa densa
model.add(Dense(128, activation="relu")) # 128 neuronas totalmente conectadas.
model.add(Dropout(0.50)) # Apaga aleatoriamente 50% de neuronas en entrenamiento.  # Evita sobreajuste.

# Salida
model.add(Dense(1, activation="sigmoid"))  # Una neurona final para clasificación binaria. valor cercano a 0 = perro. valor cercano a 1 = gato

# ============================================================
# PASO 6. COMPILACIÓN
# ============================================================
model.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    metrics=["accuracy"]
)


# model.summary()
# Imprime la arquitectura completa del modelo:
# capas, dimensiones de salida y cantidad de parámetros entrenables.

model.summary()

# ============================================================
# PASO 7. CALLBACKS
# ============================================================
# Observa el val_loss (error en validación) Si durante 7 épocas seguidas no mejora,detiene el entrenamiento automáticamente.
# Esto evita sobreajuste y pérdida de tiempo.
# restore_best_weights=True Al finalizar, recupera los pesos de la mejor época, no los de la última.

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

# Si el val_loss se estanca durante 2 épocas,reduce el learning rate a la mitad.
# Ejemplo: 0.0001 -> 0.00005 -> 0.000025. Esto permite ajustes más finos cuando el modelo ya no mejora fácilmente.
# min_lr=1e-6 evita que baje demasiado.

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

# Guarda automáticamente el mejor modelo encontradodurante el entrenamiento. monitor="val_loss" Se fija en el error de validación.
# mode="min" Busca el valor más pequeño.
# save_best_only=True Solo guarda cuando mejora.
# save_weights_only=False Guarda arquitectura + pesos completos del modelo.
# filepath=ruta_modelo Ruta donde se almacena el archivo.
# verbose=1 significa que el proceso mostrará mensajes en pantalla mientras se ejecuta.
checkpoint = ModelCheckpoint(
    filepath=ruta_modelo,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    mode="min",
    verbose=1
)

# ============================================================
# PASO 8. ENTRENAMIENTO
# ============================================================
# Esta sección inicia el entrenamiento de la red neuronal, donde el modelo aprende patrones para diferenciar perros y gatos.
# batch_size=32 El modelo procesa 32 imágenes por cada actualización de pesos.
# Si hay 3200 imágenes: 3200 / 32 = 100 batches por época. 
# epochs=30 El dataset completo puede recorrerse hasta 30 veces.

history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=30,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr, checkpoint],
    class_weight=class_weight_dict,
    verbose=1
)

# ============================================================
# PASO 8.1 CARGAR EL MEJOR MODELO
# ============================================================
# Esta parte carga desde disco el mejor modelo que fue guardado
# automáticamente durante el entrenamiento mediante ModelCheckpoint.

# No necesariamente carga la última época entrenada,
# sino la época con menor val_loss (mejor desempeño en validación).

# Esto asegura usar la versión más eficiente del modelo
# para evaluación final o predicciones nuevas.

# ruta_modelo contiene la ubicación del archivo guardado.

print("\nCargando el mejor modelo guardado...")
model = load_model(ruta_modelo)

# ============================================================
# PASO 9. EVALUACIÓN FINAL EN TEST
# ============================================================
# Evalúa el modelo usando el conjunto TEST,datos nunca vistos durante entrenamiento.
# Devuelve:test_loss: error final del modelo.
# test_acc: porcentaje de aciertos en test.

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("\nTest loss:", test_loss)
print("Test accuracy:", test_acc)


# Genera probabilidades entre 0 y 1 para cada imagen.
# Ejemplo: 0.12 = perro probable. 0.91 = gato probable

y_pred_prob = model.predict(X_test)

# Convierte probabilidades en clases finales usando umbral 0.50Si probabilidad >= 0.50 -> gato (1) Si probabilidad < 0.50 -> perro (0)
# ravel() lo deja en vector plano.

y_pred = (y_pred_prob >= 0.50).astype(int).ravel()

# Calcula precisión total manualmente. Fórmula: Accuracy = aciertos / total

acc = accuracy_score(y_test, y_pred)
print("Accuracy score (sklearn):", acc)

# Muestra aciertos y errores por clase. Estructura binaria: [[TN FP] [FN TP]]
# TN = perros bien clasificados FP = perros confundidos como gatos
# FN = gatos confundidos como perros TP = gatos bien clasificados

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

# Muestra métricas por clase:
# Precision: de lo que predijo como esa clase, cuántos eran correctos.
# Recall: de los reales de esa clase, cuántos detectó.
# F1-score: balance entre precision y recall.
# Support: cantidad real de ejemplos por clase.

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred, target_names=["PERRO", "GATO"]))

# ============================================================
# PASO 10. MODELO FINAL
# ============================================================
# Esta parte solo informa la ubicación donde quedó almacenado el mejor modelo entrenado.

print("\nEl mejor modelo quedó guardado en:", ruta_modelo)

Shape total de X: (7390, 224, 224, 3)
Shape total de y: (7390,)
Total PERROS: 4990
Total GATOS: 2400

Distribución final:
Train: (5173, 224, 224, 3) (5173,)
Validation: (1108, 224, 224, 3) (1108,)
Test: (1109, 224, 224, 3) (1109,)

Clases por conjunto:
Train -> PERROS: 3493  | GATOS: 1680
Validation -> PERROS: 748  | GATOS: 360
Test -> PERROS: 749  | GATOS: 360

Class weights: {0: np.float64(0.7404809619238477), 1: np.float64(1.5395833333333333)}


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential (Sequential)              │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 224, 224, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 224, 224, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 112, 112, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 112, 112, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 112, 112, 64)        │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 56, 56, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 56, 56, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 56, 56, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 28, 28, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 28, 28, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 28, 28, 256)         │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 14, 14, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 14, 14, 512)         │       1,180,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 14, 14, 512)         │           2,048 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_4 (MaxPooling2D)       │ (None, 7, 7, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 7, 7, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 25088)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │       3,211,3

 Total params: 4,784,065 (18.25 MB)

 Trainable params: 4,782,081 (18.24 MB)

 Non-trainable params: 1,984 (7.75 KB)

Epoch 1/30
162/162 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6013 - loss: 0.9965
Epoch 1: val_loss improved from None to 2.57757, saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\5capas_modelo_fase1_perro_gato.keras

Epoch 1: finished saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\5capas_modelo_fase1_perro_gato.keras
162/162 ━━━━━━━━━━━━━━━━━━━━ 396s 2s/step - accuracy: 0.6230 - loss: 0.7783 - val_accuracy: 0.3249 - val_loss: 2.5776 - learning_rate: 1.0000e-04
Epoch 2/30
162/162 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6502 - loss: 0.6322
Epoch 2: val_loss improved from 2.57757 to 1.23683, saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\5capas_modelo_fase1_perro_gato.ker